# 12 — Network Analysis

**Purpose:** Transform the 438 (ligand, receptor, strip) L(r) statistics from the HPC run into a
network representation of co-clustering interactions, detect gene communities via modularity
maximisation, and identify what differs between the infected strip (strip 2) and the two controls
(strips 1 and 3).

**This is a scaffold notebook** — every section is heavily annotated to explain the concept,
the library call, and the methodological choice being made. The goal is to make each step
transparent before we tighten things up for the final dissertation figures.

## The journey from raw stats to a network

We have 438 rows, each describing one (ligand, receptor, strip) pair. Each row contains:
- A 50-point observed L(r) curve (`l_obs`) — the actual measured co-clustering at radii 5–250 px
- Upper and lower permutation envelopes (`l_hi`, `l_lo`) — what L(r) looks like under spatial
  randomness (99 permutations)

The problem: we can't draw a network with 50 numbers per pair. We need to reduce each pair to a
yes/no decision (is this interaction real?) and a single magnitude (how strong is it?). The
sections below do exactly that, in order:

1. **Load data** — bring back the L(r) arrays and compute SES scores
2. **Filter** — keep only pairs where L(r) persistently exceeds the envelope
3. **Edge table** — distil each significant pair to one row with weight and scale
4. **Build graphs** — one NetworkX graph per strip
5. **Hub genes** — which genes appear in the most interactions?
6. **Modularity maximisation** — find communities of genes that interact more with each other than chance
7. **Visualise** — draw the three networks side-by-side with community colouring
8. **Spatial scale** — at what distances do significant interactions operate?
9. **Strip comparison** — which edges are infection-specific?
10. **PCA / UMAP scaffold** — placeholder for profile-level dimensionality reduction

---
**Inputs:**
- `results/lr_panel_results.parquet`
- `results/lr_panel_results.parquet.r_vals.npy`

**Outputs:**
- `results/figures/12_networks_per_strip.png`
- `results/figures/12_hub_genes.png`
- `results/figures/12_scale_distribution.png`
- `results/12_edge_summary.csv`
- `results/12_community_labels.csv`

## 0. Imports and constants

In [ ]:
# ── Standard numerical stack ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path
import itertools          # used in Section 9 for set comparisons between strips

# ── Plotting ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns

# ── Graph library ──────────────────────────────────────────────────────────
# networkx is the standard Python graph library.  It stores graphs as objects
# with nodes and edges, each of which can carry arbitrary attribute dicts.
# networkx.community is its submodule for community-detection algorithms.
import networkx as nx
from networkx.community import (
    greedy_modularity_communities,   # Clauset-Newman-Moore greedy algorithm
    louvain_communities,             # Louvain algorithm (built into nx 3.x)
    modularity,                      # function to compute the Q score of a partition
)

# ── Dimensionality reduction (used in Section 10) ─────────────────────────
# PCA: linear projection that finds the directions of greatest variance
from sklearn.decomposition import PCA
# UMAP: non-linear embedding that preserves local neighbourhood structure
import umap

# ── Paths ──────────────────────────────────────────────────────────────────
RESULTS = Path('../results/lr_panel_results.parquet')
RVALS   = Path('../results/lr_panel_results.parquet.r_vals.npy')
FIGS    = Path('../results/figures')
FIGS.mkdir(parents=True, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

# ── Strip colour palette ───────────────────────────────────────────────────
# Strip 1 and 3 are controls (blue family); strip 2 is infected (orange)
STRIP_COLORS = {'strip_1': '#4C72B0', 'strip_2': '#DD8452', 'strip_3': '#55A868'}
STRIP_LABELS = {'strip_1': 'Strip 1 (control)', 'strip_2': 'Strip 2 (infected)',
                'strip_3': 'Strip 3 (control)'}

## 1. Data loading and what the numbers mean

### What is L(r)?

Ripley's L(r) is a transformed version of the bivariate K-function. For a pair of transcript
species A (ligand) and B (receptor), L(r) at radius r tells you: **if you draw a circle of radius
r around every A transcript, how many B transcripts do you find on average — relative to how many
you'd expect if B were distributed completely randomly?**

- `l_obs` > 0: more B-transcripts near A than chance → attraction / co-clustering
- `l_obs` ≈ 0: as many B-transcripts near A as chance → spatial independence
- `l_obs` < 0: fewer B-transcripts near A than chance → segregation

We have 50 values of r from 5 px to 250 px (≈ 0.9 µm to 45 µm), so each pair gives us a
**curve** — the co-clustering signal across spatial scales.

### What are the envelopes?

`l_lo` and `l_hi` are the minimum and maximum L(r) values observed across 99 permutations in
which the gene labels were randomly shuffled. They define a **pointwise null envelope**: if the
real `l_obs` stays inside this band at every r, there is no evidence of non-random spatial
structure.

### Why compute SES?

The Standardised Effect Size normalises `l_obs` to envelope units:

    SES(r) = (l_obs(r) - midline(r)) / halfwidth(r)

where `midline = (l_hi + l_lo) / 2` and `halfwidth = (l_hi - l_lo) / 2`.

|SES value|Meaning|
|---|---|
|SES = 0|Exactly at the midline of the envelope|
|SES = +1|Right at the upper envelope edge (p ≈ 0.01 under pointwise test)|
|SES = -1|Right at the lower envelope edge|
|SES > +1|**Outside the upper envelope — attraction signal**|
|SES < -1|**Outside the lower envelope — segregation signal**|

The **peak signed SES** (`score`) captures the strongest excursion: at whichever radius the
signal is most extreme, what is the SES?

In [ ]:
# Load the aggregated HPC results
df = pd.read_parquet(RESULTS)
r_vals = np.load(RVALS)

# The parquet stores L(r) arrays as Python lists; promote them to numpy arrays
# so we can do vectorised maths on them
for col in ('l_obs', 'l_lo', 'l_hi'):
    df[col] = df[col].apply(np.asarray)

print(f'Rows:         {len(df)}')
print(f'Unique pairs: {df[["ligand","receptor"]].drop_duplicates().shape[0]}')
print(f'Strips:       {sorted(df.strip.unique())}')
print(f'r_vals:       {len(r_vals)} points, {r_vals[0]:.1f} → {r_vals[-1]:.1f} px')
df.head(3)

In [ ]:
# ── SES calculation (reproduced from nb11) ─────────────────────────────────

mid  = df.apply(lambda r: (r.l_hi + r.l_lo) / 2,   axis=1)
half = df.apply(lambda r: (r.l_hi - r.l_lo) / 2,   axis=1)

# Guard against envelope collapse: if halfwidth ≈ 0 (all permutations gave
# identical L(r) at a radius), SES is undefined — we set it to 0.
df['ses_curve'] = [
    np.where(h > 1e-9, (obs - m) / h, 0.0)
    for obs, m, h in zip(df.l_obs, mid, half)
]

def peak_signed(ses):
    """Return the SES value at the radius of greatest |SES|."""
    i = int(np.argmax(np.abs(ses)))
    return float(ses[i])

df['score']     = df.ses_curve.apply(peak_signed)
df['r_at_peak'] = df.ses_curve.apply(
    lambda s: float(r_vals[int(np.argmax(np.abs(s)))])
)

print(df[['ligand','receptor','strip','score','r_at_peak']].describe().round(2))

## 2. Significance filtering: from 438 stats to meaningful edges

### Why consecutive exceedances?

A single point where `l_obs > l_hi` is not convincing — with 50 radii and a pointwise envelope
constructed from 99 permutations, we'd expect a few spurious exceedances by chance even for a
completely random pair. What matters is **sustained** exceedance: if the observed curve is above
the upper envelope for 3 or more consecutive radii, the interaction is real across a meaningful
range of spatial scales.

**Why only filter for attraction (l_obs > l_hi)?**  
For a ligand–receptor interaction network, we care about pairs that are co-clustered: the ligand
and receptor transcripts are closer together than expected, suggesting they are being expressed in
neighbouring or overlapping cell populations. Segregation (l_obs < l_lo) is biologically
interesting too, but it means the pair is *anti-correlated* — an edge in a network of
interactions wouldn't capture that well. We flag depletion separately for completeness.

**Threshold choice:**  
`n_consec = 3` is a pragmatic choice for the POC run. After the main analysis is finalised, run
a sensitivity check with n_consec = 2 (more permissive) and n_consec = 5 (stricter) to confirm
the key results are stable.

In [ ]:
def max_consecutive_true(arr):
    """Return the maximum run length of True values in a boolean array."""
    count = max_count = 0
    for v in arr:
        count = count + 1 if v else 0
        max_count = max(max_count, count)
    return max_count

def is_significant(l_obs, l_hi, n_consec=3):
    """
    True if the observed L(r) exceeds the upper permutation envelope for at
    least n_consec consecutive radii — i.e., a spatially sustained attraction
    signal rather than a noise spike at a single scale.
    """
    above = l_obs > l_hi          # boolean array, length = len(r_vals)
    return max_consecutive_true(above) >= n_consec

def is_depleted(l_obs, l_lo, n_consec=3):
    """Same logic for segregation (below lower envelope)."""
    below = l_obs < l_lo
    return max_consecutive_true(below) >= n_consec

# ── Apply filters ──────────────────────────────────────────────────────────
N_CONSEC = 3  # design choice; see note above

df['sig_attraction'] = df.apply(
    lambda r: is_significant(r.l_obs, r.l_hi, N_CONSEC), axis=1
)
df['sig_depletion'] = df.apply(
    lambda r: is_depleted(r.l_obs, r.l_lo, N_CONSEC), axis=1
)

# ── Summary table ──────────────────────────────────────────────────────────
summary = (df.groupby('strip')
             .agg(total=('sig_attraction','count'),
                  n_attraction=('sig_attraction','sum'),
                  n_depletion=('sig_depletion','sum'))
             .assign(pct_attraction=lambda x: (x.n_attraction/x.total*100).round(1)))

print('Significance summary (n_consec =', N_CONSEC, ')')
print(summary)
print()
print('Total significant (attraction):', df.sig_attraction.sum())

In [ ]:
# Visualise how many pairs survive per strip
fig, ax = plt.subplots(figsize=(6, 3.5))

strips = ['strip_1', 'strip_2', 'strip_3']
counts = [summary.loc[s, 'n_attraction'] for s in strips]
total  = [summary.loc[s, 'total']        for s in strips]
colors = [STRIP_COLORS[s]                for s in strips]

bars = ax.bar([STRIP_LABELS[s] for s in strips], counts, color=colors, edgecolor='white')
ax.bar([STRIP_LABELS[s] for s in strips], total, color=colors, alpha=0.2, edgecolor='none')

# Annotate with exact counts
for bar, n, t in zip(bars, counts, total):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{n}/{t}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Significant pairs (attraction)')
ax.set_title(f'Pairs with ≥{N_CONSEC} consecutive r-exceedances above envelope\n'
             f'(light bar = all 146 pairs; solid = significant)')
ax.set_ylim(0, max(total) * 1.15)
plt.tight_layout()
plt.show()

## 3. Building the edge summary table

Now that we know which pairs are significant, we distil each one down to a single row in an
**edge table** — the tidy representation of what will become network edges.

### Edge weight: peak l_obs vs peak SES

Two candidates for the edge weight:

| Metric | Pros | Cons |
|---|---|---|
| **Peak `l_obs`** | Physically meaningful (px units); represents absolute co-clustering magnitude | Depends on transcript counts — a pair with 5× more transcripts could have a higher l_obs even with the same spatial pattern |
| **Peak SES** | Normalised; comparable across pairs with different abundance | Dimensionless; graph layout algorithms treat weights as forces, and SES differences may be small |

**Choice here: peak `l_obs`.** This makes the edge weight proportional to the actual spatial
clustering signal. For the dissertation results, consider running the network with both and
checking whether community membership changes.

The `r_at_peak` column is kept because it tells us the *spatial scale* of each interaction —
used in Section 8.

In [ ]:
# Keep only attraction-significant pairs
sig = df[df.sig_attraction].copy()

# For edge weight, take the l_obs value at the same radius as the SES peak
# (consistent with how r_at_peak and score are defined)
sig['weight'] = sig.apply(
    lambda r: float(r.l_obs[int(np.argmax(np.abs(r.ses_curve)))]),
    axis=1
)

# Construct the clean edge table
edges = sig[['ligand','receptor','strip','weight','score','r_at_peak']].copy()
edges = edges.rename(columns={'ligand':'source','receptor':'target','score':'ses_peak'})
edges = edges.sort_values(['strip','weight'], ascending=[True,False]).reset_index(drop=True)

print(f'Edge table shape: {edges.shape}')
print()
print('Top 10 edges by weight:')
print(edges.head(10).to_string(index=False))

# Save edge table so other notebooks / scripts can load it directly
edges.to_csv('../results/12_edge_summary.csv', index=False)
print('\nSaved → results/12_edge_summary.csv')

## 4. NetworkX: what a graph is and how we build one

### The concept

A graph `G` in NetworkX is an object that holds:
- **Nodes**: anything hashable (we use gene name strings)
- **Edges**: pairs of nodes, each with an optional attribute dictionary
- **Node attributes**: a dictionary attached to each node (e.g. `role`, `community`)
- **Edge attributes**: a dictionary attached to each edge (e.g. `weight`, `ses`, `r_at_peak`)

For our data:  
- Each **node** = one gene name (could be a ligand, a receptor, or both)
- Each **edge** = a significant (ligand, receptor) co-clustering pair in that strip
- **Edge weight** = peak l_obs (stronger co-clustering = heavier edge)

We build **one graph per strip** because the spatial structure may differ between infected
and control tissue. Comparing three separate graphs lets us identify which interactions
are consistent (strips 1 and 3) versus infection-specific (strip 2).

### Toy demo first

Before touching the real data, here is a minimal example to show what the API looks like:

In [ ]:
# ── Toy graph demo ─────────────────────────────────────────────────────────
# Imagine 5 genes and 4 detected interactions.

G_toy = nx.Graph()   # undirected graph — L-R co-clustering is symmetric

# Add nodes manually (optional — add_edge creates them automatically,
# but explicit addition lets us attach attributes upfront)
G_toy.add_node('GENE_A', role='ligand')
G_toy.add_node('GENE_B', role='receptor')
G_toy.add_node('GENE_C', role='both')
G_toy.add_node('GENE_D', role='receptor')
G_toy.add_node('GENE_E', role='ligand')

# Add edges with attributes
G_toy.add_edge('GENE_A', 'GENE_B', weight=45.2, r_at_peak=30)
G_toy.add_edge('GENE_A', 'GENE_C', weight=12.0, r_at_peak=80)
G_toy.add_edge('GENE_C', 'GENE_D', weight=33.7, r_at_peak=15)
G_toy.add_edge('GENE_E', 'GENE_D', weight=8.1,  r_at_peak=200)

# Inspect
print(f'Nodes ({G_toy.number_of_nodes()}): {list(G_toy.nodes)}')
print(f'Edges ({G_toy.number_of_edges()}):')
for u, v, attrs in G_toy.edges(data=True):
    print(f'  {u} — {v}  {attrs}')

# Access attributes
print()
print("GENE_A's role:",  G_toy.nodes['GENE_A']['role'])
print("GENE_A–B weight:", G_toy['GENE_A']['GENE_B']['weight'])

# Degree = number of edges connected to each node
print()
print('Degrees:', dict(G_toy.degree()))

In [ ]:
# ── Build the real per-strip graphs ────────────────────────────────────────

def build_strip_graph(edges_df, strip):
    """
    Build a weighted NetworkX graph for one strip.

    Nodes: all genes that appear as source or target in significant pairs.
    Edges: one per significant (source, target) pair.
    Edge attrs: weight, ses_peak, r_at_peak.
    Node attrs: role (ligand / receptor / both).
    """
    sub = edges_df[edges_df.strip == strip]
    G = nx.Graph()

    for _, row in sub.iterrows():
        G.add_edge(
            row.source, row.target,
            weight=row.weight,
            ses=row.ses_peak,
            r_at_peak=row.r_at_peak,
        )

    # Annotate each node with its role in this strip's network
    ligands   = set(sub.source)
    receptors = set(sub.target)
    for node in G.nodes():
        if node in ligands and node in receptors:
            G.nodes[node]['role'] = 'both'
        elif node in ligands:
            G.nodes[node]['role'] = 'ligand'
        else:
            G.nodes[node]['role'] = 'receptor'

    return G


strips = ['strip_1', 'strip_2', 'strip_3']
graphs = {s: build_strip_graph(edges, s) for s in strips}

# ── Summary statistics ─────────────────────────────────────────────────────
print(f"{'Strip':<12} {'Nodes':>6} {'Edges':>6} {'Density':>9}")
print('-' * 38)
for s in strips:
    G = graphs[s]
    print(f"{STRIP_LABELS[s]:<12} {G.number_of_nodes():>6} "
          f"{G.number_of_edges():>6} {nx.density(G):>9.3f}")

## 5. Graph topology: hub genes

The **degree** of a node is the number of edges connected to it — in biological terms, how many
other genes this gene has a significant spatial co-clustering interaction with.

A **hub gene** appears in many interactions. Biologically, hubs are often:
- Highly expressed cell-type markers that are present everywhere and thus physically close to
  many other transcripts (a potential confound — worth checking)
- True co-regulatory factors: a signalling molecule whose presence in a region correlates with
  many downstream partners
- Genes that serve both ligand and receptor roles in different interactions (note the `role`
  attribute we set in step 4)

Comparing hub gene rankings across strips tells us: which signalling genes are constitutively
active (high degree in all strips) vs which are uniquely or differentially active in the
infected strip?

In [ ]:
TOP_N = 15

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, s in zip(axes, strips):
    G = graphs[s]
    deg = pd.Series(dict(G.degree())).sort_values(ascending=False)
    top = deg.head(TOP_N)

    # Colour bars by node role for quick interpretation
    role_colors = {
        'ligand':   '#4C72B0',
        'receptor': '#DD8452',
        'both':     '#55A868',
    }
    bar_colors = [role_colors.get(G.nodes[n].get('role','receptor'), 'grey')
                  for n in top.index]

    ax.barh(range(len(top)), top.values, color=bar_colors, edgecolor='white')
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top.index, fontsize=9)
    ax.invert_yaxis()   # highest degree at top
    ax.set_xlabel('Degree (# significant partners)')
    ax.set_title(STRIP_LABELS[s], color=STRIP_COLORS[s], fontweight='bold')
    ax.axvline(1, color='grey', lw=0.5, ls='--')

# Shared legend for role colours
handles = [mpatches.Patch(color=c, label=r) for r, c in role_colors.items()]
fig.legend(handles=handles, title='Node role', loc='lower center',
           ncol=3, bbox_to_anchor=(0.5, -0.05))
plt.suptitle(f'Top {TOP_N} hub genes per strip', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / '12_hub_genes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/figures/12_hub_genes.png')

## 6. Modularity maximisation: how it works and how we run it

### What is modularity Q?

Given a partition of the graph into communities C₁, C₂, …, Cₖ, the **modularity Q** measures
whether the fraction of edges *within* communities exceeds the fraction expected under a null
model that preserves the degree sequence (i.e. each node has the same number of edges as in the
real graph, but they are rewired randomly):

    Q = Σ_c [ (e_c / m) - (d_c / 2m)² ]

where `e_c` = edges within community c, `m` = total edges, `d_c` = sum of degrees of nodes in c.

**Interpretation:**
- Q ≈ 0: no more within-community edges than expected by chance — no community structure
- Q > 0.3: meaningful community structure (rough empirical threshold)
- Q → 1: perfect communities (never achieved in practice)
- Q can be negative if the partition is worse than the null model

For a biological ligand–receptor network, Q > 0.3 would mean the genes naturally group into
clusters that interact internally more than externally — analogous to signalling pathways or
cell-type-specific communication circuits.

### Algorithm 1: Clauset-Newman-Moore (greedy)

1. Start: each node is its own community → Q = 0 (no within-community edges at all)
2. For every pair of communities, compute ΔQ from merging them
3. Merge the pair with the highest ΔQ
4. Repeat until no merge improves Q

This is **fast** (O(m d log n)) and deterministic. It tends to find fewer, larger communities
than Louvain, and can miss fine-grained structure in dense graphs.

NetworkX call: `greedy_modularity_communities(G, weight='weight')`

### Algorithm 2: Louvain

1. Start: each node is its own community
2. **Phase 1 (local moves):** for each node (in random order), compute ΔQ from moving it into
   each of its neighbours' communities; accept the best move; repeat until no node moves
3. **Phase 2 (aggregation):** collapse each community into a single super-node, keeping edge
   weights as the sum of internal edges; go back to Phase 1 on the condensed graph
4. Repeat phases until Q stops improving

This is **higher quality** than greedy (typically finds higher Q) and handles multi-scale
community structure. It is **non-deterministic** because the random node order in Phase 1 affects
the result — always fix `seed=42` for reproducibility. For a dissertation-quality result, run
5–10 seeds and report the partition with the highest Q.

NetworkX call: `louvain_communities(G, weight='weight', seed=42)`

### Which to use?

We will use **Louvain** for the final figures because it gives higher Q and handles the weighted
graph better. The greedy result is shown here for comparison.

In [ ]:
# ── Run both algorithms on strip 2 (infected) and compare ──────────────────
G2 = graphs['strip_2']

if G2.number_of_edges() < 2:
    print('WARNING: strip_2 graph has fewer than 2 edges — community detection is trivial.')
    print('Check the significance threshold or run with more permutations (n_sim=199).')
else:
    comms_greedy  = list(greedy_modularity_communities(G2, weight='weight'))
    comms_louvain = list(louvain_communities(G2,          weight='weight', seed=42))

    q_greedy  = modularity(G2, comms_greedy,  weight='weight')
    q_louvain = modularity(G2, comms_louvain, weight='weight')

    print(f'Strip 2 — Greedy  Q = {q_greedy:.3f}  ({len(comms_greedy)} communities)')
    print(f'Strip 2 — Louvain Q = {q_louvain:.3f}  ({len(comms_louvain)} communities)')
    print()
    print('Greedy communities:')
    for i, c in enumerate(sorted(comms_greedy, key=len, reverse=True)):
        print(f'  C{i}: {sorted(c)}')
    print()
    print('Louvain communities:')
    for i, c in enumerate(sorted(comms_louvain, key=len, reverse=True)):
        print(f'  C{i}: {sorted(c)}')

In [ ]:
# ── Assign community labels to all three strip graphs ──────────────────────
# We store an integer community label on each node as a node attribute.
# Communities are sorted largest-first so C0 is always the biggest community.

community_records = []   # collect for the output CSV

for s in strips:
    G = graphs[s]

    if G.number_of_edges() < 2:
        # Trivial case: every node is its own community
        for node in G.nodes():
            G.nodes[node]['community'] = 0
        print(f'{s}: too few edges for meaningful community detection')
        continue

    comms = list(louvain_communities(G, weight='weight', seed=42))
    # Sort by size so the label integers are consistent across runs
    comms = sorted(comms, key=len, reverse=True)

    q = modularity(G, comms, weight='weight')
    print(f'{STRIP_LABELS[s]}: Q = {q:.3f}, {len(comms)} communities')

    for comm_id, comm_set in enumerate(comms):
        for gene in comm_set:
            G.nodes[gene]['community'] = comm_id
            community_records.append({
                'gene': gene,
                'strip': s,
                'community': comm_id,
                'role': G.nodes[gene].get('role','unknown'),
                'degree': G.degree(gene),
                'modularity_Q': q,
            })

# Save community labels
comm_df = pd.DataFrame(community_records)
comm_df.to_csv('../results/12_community_labels.csv', index=False)
print('\nSaved → results/12_community_labels.csv')
comm_df.head(10)

## 7. Network visualisation per strip

### Layout algorithm: spring (Fruchterman-Reingold)

Nodes are positioned as if connected by springs: connected nodes attract each other, and all
nodes repel each other (like charged particles). The system is simulated until it reaches a low
energy configuration. We use edge **weights** to modulate the spring strength — strongly
co-clustered pairs pull their nodes closer together, so genes with strong interactions end up
spatially close in the plot.

Always pass `seed=42` to `spring_layout` — without a seed the initial node positions are random
and the layout changes on every run.

### Visual encoding

| Visual channel | Data variable | Why |
|---|---|---|
| Node colour | Community label | Groups co-interacting genes visually |
| Node size | Degree | Hub genes are prominent |
| Node shape | (all circles — simplicity) | — |
| Edge width | Normalised weight | Stronger co-clustering = thicker line |
| Edge colour | (light grey) | Avoids distraction from node colour |
| Edge alpha | 0.5 | Dense graphs would be unreadable at alpha=1 |

In [ ]:
# ── Choose a colourmap for communities ────────────────────────────────────
# 'tab10' has 10 distinct colours, enough for typical community counts.
# If a strip has > 10 communities, switch to 'tab20'.
CMAP = cm.get_cmap('tab10')

def draw_strip_network(ax, G, title, title_color='black'):
    """
    Draw the network for one strip on the given Axes.
    Returns the layout dict so it can be reused for annotating specific nodes.
    """
    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'No significant pairs', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(title, color=title_color, fontweight='bold')
        return {}

    # ── Spring layout ──────────────────────────────────────────────────────
    # k controls ideal spring length; smaller k = more compact layout.
    # If the graph is very small, spring_layout may be degenerate — add a
    # fallback to kamada_kawai_layout for < 5 nodes.
    if G.number_of_nodes() < 5:
        pos = nx.kamada_kawai_layout(G, weight='weight')
    else:
        pos = nx.spring_layout(G, weight='weight', seed=42, k=1.5)

    # ── Node colours from community label ─────────────────────────────────
    n_comms = max((G.nodes[n].get('community', 0) for n in G.nodes()), default=0) + 1
    node_colors = [CMAP(G.nodes[n].get('community', 0) / max(n_comms - 1, 1))
                   for n in G.nodes()]

    # ── Node sizes scaled by degree ────────────────────────────────────────
    degrees = np.array([G.degree(n) for n in G.nodes()], dtype=float)
    # Normalise to [100, 800] px²
    if degrees.max() > degrees.min():
        node_sizes = 100 + 700 * (degrees - degrees.min()) / (degrees.max() - degrees.min())
    else:
        node_sizes = np.full(len(degrees), 300)

    # ── Edge widths scaled by weight ───────────────────────────────────────
    weights = np.array([G[u][v]['weight'] for u, v in G.edges()], dtype=float)
    if weights.max() > weights.min():
        edge_widths = 0.5 + 3.5 * (weights - weights.min()) / (weights.max() - weights.min())
    else:
        edge_widths = np.ones(len(weights)) * 1.5

    # ── Draw ───────────────────────────────────────────────────────────────
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_color=node_colors,
                           node_size=node_sizes,
                           alpha=0.9)

    nx.draw_networkx_edges(G, pos, ax=ax,
                           width=edge_widths,
                           alpha=0.5,
                           edge_color='#888888')

    nx.draw_networkx_labels(G, pos, ax=ax,
                            font_size=7,
                            font_weight='bold')

    ax.set_title(title, color=title_color, fontweight='bold', fontsize=11)
    ax.axis('off')
    return pos


# ── Draw all three strips ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, s in zip(axes, strips):
    draw_strip_network(ax, graphs[s], title=STRIP_LABELS[s], title_color=STRIP_COLORS[s])

# ── Shared legend for community colours ──────────────────────────────────
# Determine max community count across all strips
max_comms = max(
    max((graphs[s].nodes[n].get('community', 0) for n in graphs[s].nodes()), default=0)
    for s in strips
) + 1

comm_handles = [
    mpatches.Patch(color=CMAP(i / max(max_comms - 1, 1)), label=f'Community {i}')
    for i in range(max_comms)
]
fig.legend(handles=comm_handles, title='Community', loc='lower center',
           ncol=min(max_comms, 5), bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Ligand–receptor co-clustering networks per strip\n'
             '(node size = degree, edge width = interaction strength)',
             y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / '12_networks_per_strip.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/figures/12_networks_per_strip.png')

## 8. Spatial scale of interactions: r_at_peak analysis

The `r_at_peak` value tells us **at what spatial radius the interaction is strongest**. Because
the CosMx platform has a pixel size of ≈ 0.18 µm, we can convert:

| Band | r range (px) | r range (µm) | Biological interpretation |
|---|---|---|---|
| Short | < 50 px | < 9 µm | Direct cell–cell contact; within a single cell |
| Mid | 50 – 150 px | 9 – 27 µm | Paracrine: one or a few cell diameters away |
| Long | > 150 px | > 27 µm | Tissue-level gradient; signalling over multiple cells |

**Why does scale matter for the strip comparison?**  
If infection changes the *spatial scale* of interactions — not just which interactions occur —
that would suggest a reorganisation of tissue architecture or a shift from juxtacrine to
paracrine signalling.

In [ ]:
# ── Define scale bands ────────────────────────────────────────────────────
def assign_band(r_px):
    if r_px < 50:
        return 'short (<9 µm)'
    elif r_px <= 150:
        return 'mid (9–27 µm)'
    else:
        return 'long (>27 µm)'

edges['scale_band'] = edges.r_at_peak.apply(assign_band)

# ── Summary table ─────────────────────────────────────────────────────────
band_summary = (edges.groupby(['strip','scale_band'])
                  .size()
                  .unstack(fill_value=0)
                  [['short (<9 µm)', 'mid (9–27 µm)', 'long (>27 µm)']])  # fix order
print('Interaction counts by spatial scale and strip:')
print(band_summary)

# ── Histogram ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, s in zip(axes, strips):
    sub = edges[edges.strip == s]
    ax.hist(sub.r_at_peak, bins=20, range=(0, 260),
            color=STRIP_COLORS[s], edgecolor='white', alpha=0.85)

    # Mark band boundaries
    ax.axvline(50,  color='grey', lw=1, ls='--', alpha=0.7, label='50 px (9 µm)')
    ax.axvline(150, color='grey', lw=1, ls=':',  alpha=0.7, label='150 px (27 µm)')

    ax.set_xlabel('r at peak (px)')
    ax.set_title(STRIP_LABELS[s], color=STRIP_COLORS[s], fontweight='bold')

axes[0].set_ylabel('# significant pairs')
axes[0].legend(fontsize=8)

# Add secondary x-axis label showing µm scale
for ax in axes:
    ax2 = ax.twiny()
    ax2.set_xlim(np.array(ax.get_xlim()) * 0.18)
    ax2.set_xlabel('r at peak (µm)', fontsize=8)

plt.suptitle('Spatial scale of significant interactions', y=1.05, fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / '12_scale_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/figures/12_scale_distribution.png')

## 9. Strip 2 vs controls: differential edge scaffold

The core scientific question: **which interactions are infection-specific?**

We define four categories of edge:

| Category | Definition | Interpretation |
|---|---|---|
| **Constitutive** | Significant in strip 1, 2, AND 3 | Present regardless of infection state |
| **Control-only** | Significant in BOTH strip 1 AND 3, absent in strip 2 | Suppressed by infection |
| **Infection-specific** | Significant in strip 2, absent in BOTH controls | Induced by infection |
| **Mixed** | Everything else (one control + infected, one control only, etc.) | Inconsistent across replicates |

**Methodological decision point:**  
Requiring both controls to agree (`BOTH strip_1 AND strip_3`) is strict — any technical noise in
one control strip would move a pair into 'mixed'. An alternative is to require EITHER control.
This choice affects the infection-specific edge count; flag it for sensitivity analysis.

**Note on strip 3 polygon:**  
The custom window for strip 3 was flagged as needing a redraw (it misses a KRT co-expression
site). This may reduce sensitivity in strip 3. Any pair absent only from strip 3 should be
treated cautiously until the polygon is updated.

In [ ]:
# ── Build edge sets per strip ─────────────────────────────────────────────
# Each 'edge' is a frozenset {source, target} — undirected, so order doesn't matter.
# (If we later decide to make the graph directed, change this to tuples.)

def edge_set(strip_name):
    sub = edges[edges.strip == strip_name]
    return {frozenset({r.source, r.target}) for _, r in sub.iterrows()}

e1 = edge_set('strip_1')
e2 = edge_set('strip_2')
e3 = edge_set('strip_3')

# ── Categorise ───────────────────────────────────────────────────────────
constitutive       = e1 & e2 & e3                  # in all three strips
control_robust     = (e1 & e3) - e2                # both controls, not infected
infection_specific = e2 - (e1 | e3)               # infected only
mixed              = (e1 | e2 | e3) - constitutive - control_robust - infection_specific

print(f'Total unique edges (union):       {len(e1 | e2 | e3)}')
print(f'Constitutive (all 3 strips):       {len(constitutive)}')
print(f'Control-only (suppressed in s2):   {len(control_robust)}')
print(f'Infection-specific (gained in s2): {len(infection_specific)}')
print(f'Mixed (inconsistent):              {len(mixed)}')

if infection_specific:
    print()
    print('Infection-specific pairs:')
    for pair in sorted(infection_specific, key=lambda x: sorted(x)[0]):
        a, b = sorted(pair)
        row = edges[(edges.strip=='strip_2') &
                    (((edges.source==a)&(edges.target==b)) |
                     ((edges.source==b)&(edges.target==a)))].iloc[0]
        print(f'  {a} × {b}  (weight={row.weight:.1f}, r_peak={row.r_at_peak:.0f} px)')

if control_robust:
    print()
    print('Control-only pairs (suppressed in strip 2):')
    for pair in sorted(control_robust, key=lambda x: sorted(x)[0]):
        a, b = sorted(pair)
        print(f'  {a} × {b}')

In [ ]:
# ── Bar chart summary ─────────────────────────────────────────────────────
categories = ['Constitutive', 'Control-only\n(suppressed)', 'Infection-\nspecific', 'Mixed']
counts     = [len(constitutive), len(control_robust), len(infection_specific), len(mixed)]
cat_colors = ['#888888', '#4C72B0', '#DD8452', '#CCCCCC']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(categories, counts, color=cat_colors, edgecolor='white')
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(n), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Number of ligand–receptor pairs')
ax.set_title('Classification of interaction edges\nrelative to infection status')
ax.set_ylim(0, max(counts) * 1.15 + 1)
plt.tight_layout()
plt.show()

# TODO: save this figure once the significance threshold is finalised

## 10. Placeholder: PCA and UMAP on L(r) profiles

In the sections above we reduced each pair to a single number (peak L or peak SES). But the
**full L(r) curve** contains information about the *shape* of the interaction — not just its
peak, but how it rises, at what scale, and whether it has multiple peaks.

Two complementary approaches to visualise this high-dimensional profile data:

**PCA (Principal Component Analysis):**
- Finds the directions of maximum variance in the profile matrix
- PC1 is likely to capture overall interaction strength (correlated with score)
- PC2 likely captures the scale shape (early vs late peak)
- Linear — fast, interpretable, but may miss non-linear structure

**UMAP (Uniform Manifold Approximation and Projection):**
- Preserves the local neighbourhood structure in 50-dimensional profile space
- Pairs with similar L(r) shapes end up close in the 2D embedding
- Coloured by community label, this can validate whether community detection
  found groups that also share similar interaction profiles

These cells are marked TODO because the optimal input matrix (which pairs to include, whether
to use SES curves or raw l_obs) should be decided after the significance threshold is finalised.

In [ ]:
# TODO: run once the significant pair count is finalised and community labels are stable.

# ── Build profile matrix ──────────────────────────────────────────────────
# Rows = significant pairs (across all strips), columns = r_vals (50 dimensions)
# Using the SES curve rather than raw l_obs so profiles are normalised and
# comparable across pairs with different transcript abundances.

profile_rows = []
profile_meta = []   # strip and pair labels for colouring

for _, row in sig.iterrows():
    profile_rows.append(row.ses_curve)
    profile_meta.append({
        'source': row.ligand,
        'target': row.receptor,
        'strip': row.strip,
        'score': row.score,
        # community label will be added below once comm_df is ready
    })

X = np.stack(profile_rows)   # shape: (n_significant, 50)
meta = pd.DataFrame(profile_meta)

# Merge community labels
meta = meta.merge(
    comm_df.rename(columns={'gene':'source'})[['source','strip','community']],
    on=['source','strip'], how='left'
)

print(f'Profile matrix shape: {X.shape}')
print(f'NaN in profiles: {np.isnan(X).sum()} (should be 0)')

In [ ]:
# TODO: uncomment and run after finalising significance threshold

# ── PCA ───────────────────────────────────────────────────────────────────
# pca = PCA(n_components=2)
# X_pca = pca.fit_transform(X)
# print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, '
#       f'PC2={pca.explained_variance_ratio_[1]:.1%}')

# fig, ax = plt.subplots(figsize=(7, 6))
# for s in strips:
#     mask = meta.strip == s
#     ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
#                c=meta.loc[mask, 'community'].fillna(0),
#                cmap='tab10', marker='o' if s == 'strip_2' else 's',
#                alpha=0.7, label=STRIP_LABELS[s], s=60)
# ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
# ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
# ax.set_title('PCA of significant L(r) SES profiles')
# ax.legend()
# plt.tight_layout()
# plt.show()

# ── UMAP ──────────────────────────────────────────────────────────────────
# reducer = umap.UMAP(n_components=2, n_neighbors=10, min_dist=0.1,
#                     metric='euclidean', random_state=42)
# X_umap = reducer.fit_transform(X)

# fig, ax = plt.subplots(figsize=(7, 6))
# for s in strips:
#     mask = meta.strip == s
#     ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
#                c=meta.loc[mask, 'community'].fillna(0),
#                cmap='tab10', alpha=0.7, label=STRIP_LABELS[s], s=60)
# ax.set_xlabel('UMAP 1')
# ax.set_ylabel('UMAP 2')
# ax.set_title('UMAP of significant L(r) SES profiles')
# ax.legend()
# plt.tight_layout()
# plt.show()

print('PCA and UMAP cells are commented out — uncomment once threshold is finalised.')
print('Input matrix X is ready with shape:', X.shape)

## Summary: what this scaffold produces

| Output | File | When to finalise |
|---|---|---|
| Network plots (all 3 strips) | `results/figures/12_networks_per_strip.png` | After significance threshold confirmed |
| Hub gene bar charts | `results/figures/12_hub_genes.png` | Same |
| Scale distribution histograms | `results/figures/12_scale_distribution.png` | Same |
| Edge summary table | `results/12_edge_summary.csv` | Same |
| Community labels | `results/12_community_labels.csv` | Same |
| PCA / UMAP plots | (not yet saved) | After Section 10 uncommented |

### Open decisions before finalising

1. **`n_consec` threshold** — currently 3; run sensitivity check with 2 and 5
2. **Edge weight** — peak `l_obs` used here; check whether SES changes community structure
3. **Louvain seeds** — run 5+ seeds; report highest-Q partition in the dissertation
4. **Strip 3 polygon** — redraw the custom window (see nb09b) before treating strip 3 as fully reliable
5. **Strip comparison rule** — BOTH controls required for 'constitutive'; consider EITHER as sensitivity check